In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

df_model = pd.read_parquet("telemetry_features_nominal_clean.parquet")
df_model = df_model.sort_values(["machine", "ts"]).reset_index(drop=True)

Définition des seuils critiques et de l’horizon et construction de la cible “panne imminente oui/non”

In [10]:
TEMP_CRIT = 35.0
POWER_CRIT = 250.0
HORIZON = 1  # 1 pas de temps

# flags instantanés
df_model["high_temp_now"] = (df_model["cputempc"] >= TEMP_CRIT).astype(int)
df_model["high_power_now"] = (df_model["totalpowerw"] >= POWER_CRIT).astype(int)

# flags futurs
df_model["high_temp_future"] = (
    df_model.groupby("machine")["high_temp_now"].shift(-HORIZON)
)
df_model["high_power_future"] = (
    df_model.groupby("machine")["high_power_now"].shift(-HORIZON)
)

# cible binaire
df_model["failure_imminent"] = (
    ((df_model["high_temp_future"] == 1) | (df_model["high_power_future"] == 1))
    .astype("Int64")
)

# enlever les lignes sans futur
df_model = df_model.dropna(subset=["failure_imminent"]).reset_index(drop=True)
df_model["failure_imminent"] = df_model["failure_imminent"].astype(int)

In [11]:
# Contrôle de l'équilibre des classes

df_model["failure_imminent"].value_counts(normalize=True)

failure_imminent
0    0.958502
1    0.041498
Name: proportion, dtype: float64

In [12]:
feature_cols = [
    c for c in df_model.columns
    if ("lag" in c or "rollmean" in c)
]

feature_cols += ["loadpercent", "cputempc", "totalpowerw", "ambientdctempc", "externaltempc"]
feature_cols = list(dict.fromkeys(feature_cols))  # dédoublonner

target_col = "failure_imminent"

X = df_model[feature_cols]
y = df_model[target_col]

In [13]:
# split 80% / 20% en respectant l'ordre temporel global
split_idx = int(0.8 * len(df_model))

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [7]:
df_model.to_parquet(
    "telemetry_features_nominal_with_target.parquet",
    index=False
)

Baseline RandomForest avec classes déséquilibrées

In [14]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",  # important pour les 4 % de positifs
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

Métriques

In [15]:
print(classification_report(y_test, y_pred, digits=3))

auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
print(f"ROC AUC: {auc:.3f}")
print(f"Average Precision (PR AUC): {ap:.3f}")

              precision    recall  f1-score   support

           0      1.000     1.000     1.000      2153
           1      0.992     1.000     0.996       117

    accuracy                          1.000      2270
   macro avg      0.996     1.000     0.998      2270
weighted avg      1.000     1.000     1.000      2270

ROC AUC: 1.000
Average Precision (PR AUC): 1.000


Attention!!!!! Résultats trop "parfaits"

In [16]:
# Corrélation simple entre la cible et les features brutes

df_tmp = df_model[["failure_imminent", "cputempc", "totalpowerw"]].corr()
print(df_tmp)

                  failure_imminent  cputempc  totalpowerw
failure_imminent          1.000000  0.744729     0.744500
cputempc                  0.744729  1.000000     0.999681
totalpowerw               0.744500  0.999681     1.000000


In [17]:
# Feature importance du RandomForest

import numpy as np

importances = pd.Series(rf.feature_importances_, index=feature_cols)
print(importances.sort_values(ascending=False).head(20))

cputempc                   0.203348
loadpercent                0.183287
loadpercent_lag1           0.129442
totalpowerw                0.110228
totalpowerw_lag1           0.102951
cputempc_lag1              0.099665
loadpercent_rollmean_5     0.053050
totalpowerw_lag3           0.039851
cputempc_rollmean_5        0.029946
loadpercent_lag3           0.023356
cputempc_lag3              0.020013
cputempc_lag6              0.003620
cputempc_rollmean_15       0.000372
totalpowerw_lag6           0.000320
loadpercent_rollmean_15    0.000288
loadpercent_lag6           0.000262
ambientdctempc             0.000000
externaltempc              0.000000
dtype: float64
